In [1]:
# Cell 1 - Import libraries

from pathlib import Path
import json
import re
import pandas as pd
import numpy as np
import folium
from branca.colormap import linear

In [2]:
# Cell 2 - Set paths automatically and define visualization parameters

from pathlib import Path
import re

BASE_DIR = Path(r"D:\JupyterProjects\tomtom_move\data\traffic_stats")
ROUTES_PARENT = BASE_DIR / "routes"
XLSX_PATH = BASE_DIR / "jobs_9031079_results_Worcester_test.xlsx"

print("BASE_DIR exists:", BASE_DIR.exists(), BASE_DIR)
print("ROUTES_PARENT exists:", ROUTES_PARENT.exists(), ROUTES_PARENT)
print("XLSX exists:", XLSX_PATH.exists(), XLSX_PATH)

# Auto-detect the folder that contains the GeoJSON files
candidate_dirs = [p for p in ROUTES_PARENT.iterdir() if p.is_dir()]
print("\nFolders under routes:")
for p in candidate_dirs:
    print("-", p.name)

ROUTE_DIR = None
for p in candidate_dirs:
    if "9031079" in p.name.lower() and "worcester" in p.name.lower():
        ROUTE_DIR = p
        break

if ROUTE_DIR is None:
    raise FileNotFoundError("Could not automatically find the GeoJSON folder under routes.")

print("\nSelected ROUTE_DIR:")
print(ROUTE_DIR)

# Auto-build route file map from filenames like ..._1.geojson, ..._2.geojson
ROUTE_FILE_MAP = {}

geojson_files = sorted(ROUTE_DIR.glob("*.geojson"))
print("\nGeoJSON files found:")
for f in geojson_files:
    print("-", f.name)
    m = re.search(r"_(\d+)\.geojson$", f.name)
    if m:
        route_id = int(m.group(1))
        ROUTE_FILE_MAP[route_id] = f.name

print("\nROUTE_FILE_MAP:")
print(ROUTE_FILE_MAP)

# Visualization parameters
METRIC_MODE = "congestion_index"
TIME_SET = "PM_Peak: 16:00-19:00"
TRAVEL_TIME_RATIO_BASE = "AM_Peak: 7:00-9:00"

GEOJSON_NEW_SEGMENT_KEYS = [
    "newSegmentId",
    "new_segment_id",
    "New Segment ID",
    "segmentUuid",
    "uuid",
]

GEOJSON_SEGMENT_KEYS = [
    "segmentId",
    "segment_id",
    "Segment ID",
    "tmcSegmentId",
    "tmc_segment_id",
]

OUTPUT_HTML = BASE_DIR / "worcester_traffic_map.html"

print("\nMETRIC_MODE:", METRIC_MODE)
print("TIME_SET:", TIME_SET)
print("Output HTML:", OUTPUT_HTML)

BASE_DIR exists: True D:\JupyterProjects\tomtom_move\data\traffic_stats
ROUTES_PARENT exists: True D:\JupyterProjects\tomtom_move\data\traffic_stats\routes
XLSX exists: False D:\JupyterProjects\tomtom_move\data\traffic_stats\jobs_9031079_results_Worcester_test.xlsx

Folders under routes:
- jobs_9031079_results_Worcester_test.geojson

Selected ROUTE_DIR:
D:\JupyterProjects\tomtom_move\data\traffic_stats\routes\jobs_9031079_results_Worcester_test.geojson

GeoJSON files found:
- chandler_street_corridor_3.geojson
- downtown_cross_town_route_5.geojson
- downtown_to_i290_connector_4.geojson
- main_street_corridoor_1.geojson
- park_avenue_corridor_2.geojson

ROUTE_FILE_MAP:
{3: 'chandler_street_corridor_3.geojson', 5: 'downtown_cross_town_route_5.geojson', 4: 'downtown_to_i290_connector_4.geojson', 1: 'main_street_corridoor_1.geojson', 2: 'park_avenue_corridor_2.geojson'}

METRIC_MODE: congestion_index
TIME_SET: PM_Peak: 16:00-19:00
Output HTML: D:\JupyterProjects\tomtom_move\data\traffic_st

In [3]:
# Cell 3 - Define helper functions for reading Excel and matching IDs

def normalize_id(x):
    """
    Normalize IDs so Excel IDs and GeoJSON IDs can be matched more reliably.
    """
    if pd.isna(x):
        return None

    s = str(x).strip()
    if not s or s.lower() == "nan":
        return None

    # Convert values like 258400000506020.0 -> 258400000506020
    if re.fullmatch(r"-?\d+\.0+", s):
        s = s.split(".")[0]

    return s.lower()


def canonical_key_name(key):
    """
    Normalize property names for fuzzy matching.
    """
    return re.sub(r"[^a-z0-9]", "", str(key).lower())


def read_route_sheet(route_id, sheet_suffix):
    """
    Read one RouteX-... sheet from the workbook.
    Header starts on Excel row 5, so pandas header index is 4.
    """
    sheet_name = f"Route{route_id}-{sheet_suffix}"
    df = pd.read_excel(
        XLSX_PATH,
        sheet_name=sheet_name,
        header=4,
        dtype={"Segment ID": str, "New Segment ID": str}
    )
    return df


def load_metric_dataframe(route_id, metric_mode, time_set):
    """
    Build one standardized DataFrame for one route and one selected metric.
    """
    speed_df = read_route_sheet(route_id, "Speeds(harmonic avg)")
    sample_df = read_route_sheet(route_id, "Sample size")
    relstd_df = read_route_sheet(route_id, "RelativeStdDeviation")
    ttr_df = read_route_sheet(route_id, "TravelTimeRatio")

    speed_col = f"Speed(kph) {time_set}"
    sample_col = f"Sample size {time_set}"
    relstd_col = f"StdDev / TravelTime(avg) {time_set}"

    result = speed_df[[
        "Segment ID",
        "New Segment ID",
        "Distance along route (m)",
        "Speed Limit(kph)",
        "Street Name"
    ]].copy()

    result["harmonic_speed_kph"] = pd.to_numeric(speed_df[speed_col], errors="coerce")
    result["sample_size"] = pd.to_numeric(sample_df[sample_col], errors="coerce")

    if metric_mode == "speed_harmonic":
        result["metric_value"] = result["harmonic_speed_kph"]
        metric_label = f"Harmonic speed (kph) | {time_set}"
        higher_is_worse = False

    elif metric_mode == "congestion_index":
        # A simple congestion score:
        # 0 means close to or above speed limit, larger means slower than speed limit
        speed_limit = pd.to_numeric(result["Speed Limit(kph)"], errors="coerce")
        result["metric_value"] = (1 - result["harmonic_speed_kph"] / speed_limit).clip(lower=0)
        metric_label = f"Congestion index = 1 - speed/speed_limit | {time_set}"
        higher_is_worse = True

    elif metric_mode == "sample_size":
        result["metric_value"] = result["sample_size"]
        metric_label = f"Sample size | {time_set}"
        higher_is_worse = False

    elif metric_mode == "relative_std_dev":
        result["metric_value"] = pd.to_numeric(relstd_df[relstd_col], errors="coerce")
        metric_label = f"Relative std. deviation | {time_set}"
        higher_is_worse = True

    elif metric_mode == "travel_time_ratio":
        if time_set == TRAVEL_TIME_RATIO_BASE:
            raise ValueError(
                "TravelTimeRatio sheet does not contain AM / AM. "
                "Choose Midday, PM_Peak, or Weekend_noon."
            )
        ttr_col = f"Ratio {time_set} / {TRAVEL_TIME_RATIO_BASE}"
        result["metric_value"] = pd.to_numeric(ttr_df[ttr_col], errors="coerce")
        metric_label = f"Travel time ratio: {time_set} / {TRAVEL_TIME_RATIO_BASE}"
        higher_is_worse = True

    else:
        raise ValueError(f"Unsupported METRIC_MODE: {metric_mode}")

    result["segment_id_norm"] = result["Segment ID"].map(normalize_id)
    result["new_segment_id_norm"] = result["New Segment ID"].map(normalize_id)

    return result, {
        "metric_label": metric_label,
        "higher_is_worse": higher_is_worse
    }


def build_route_lookups(df):
    """
    Create two lookup dictionaries:
    one for New Segment ID, one for Segment ID.
    """
    new_lookup = {}
    seg_lookup = {}

    for _, row in df.iterrows():
        record = row.to_dict()

        new_id = record.get("new_segment_id_norm")
        seg_id = record.get("segment_id_norm")

        if new_id:
            new_lookup[new_id] = record
        if seg_id:
            seg_lookup[seg_id] = record

    return new_lookup, seg_lookup


def try_match_feature(feature, new_lookup, seg_lookup):
    """
    Match one GeoJSON feature to one Excel row.
    First try explicit new-segment keys, then explicit segment keys,
    then heuristic property-name matching, then feature['id'].
    """
    props = feature.get("properties", {})

    # 1) Explicit new segment keys
    for key in GEOJSON_NEW_SEGMENT_KEYS:
        if key in props:
            candidate = normalize_id(props[key])
            if candidate in new_lookup:
                return new_lookup[candidate]

    # 2) Explicit segment keys
    for key in GEOJSON_SEGMENT_KEYS:
        if key in props:
            candidate = normalize_id(props[key])
            if candidate in seg_lookup:
                return seg_lookup[candidate]

    # 3) Heuristic key matching
    for key, value in props.items():
        ck = canonical_key_name(key)
        candidate = normalize_id(value)

        if candidate is None:
            continue

        if ("newsegment" in ck or "uuid" in ck) and candidate in new_lookup:
            return new_lookup[candidate]

        if ("segmentid" in ck or ck == "segment") and candidate in seg_lookup:
            return seg_lookup[candidate]

    # 4) Fallback to feature id
    feature_id = normalize_id(feature.get("id"))
    if feature_id in new_lookup:
        return new_lookup[feature_id]
    if feature_id in seg_lookup:
        return seg_lookup[feature_id]

    return None


def format_metric_text(metric_mode, value):
    if value is None or pd.isna(value):
        return "No data"

    if metric_mode == "speed_harmonic":
        return f"{value:.1f} kph"
    elif metric_mode == "congestion_index":
        return f"{value:.3f}"
    elif metric_mode == "sample_size":
        return f"{int(round(value))}"
    elif metric_mode == "relative_std_dev":
        return f"{value:.2f}"
    elif metric_mode == "travel_time_ratio":
        return f"{value:.2f}"
    else:
        return str(value)


def safe_text(x):
    if x is None or pd.isna(x):
        return "N/A"
    return str(x)


def collect_all_coords_from_geojson_dict(gj):
    """
    Extract all [lat, lon] pairs from a GeoJSON dict.
    """
    coords = []

    def walk(obj):
        if isinstance(obj, (list, tuple)):
            if len(obj) >= 2 and isinstance(obj[0], (int, float)) and isinstance(obj[1], (int, float)):
                coords.append((obj[1], obj[0]))  # lat, lon
            else:
                for item in obj:
                    walk(item)

    if gj.get("type") == "FeatureCollection":
        for feature in gj.get("features", []):
            geom = feature.get("geometry")
            if geom and "coordinates" in geom:
                walk(geom["coordinates"])
    elif gj.get("type") == "Feature":
        geom = gj.get("geometry")
        if geom and "coordinates" in geom:
            walk(geom["coordinates"])

    return coords

In [4]:
# Cell 4 - Preview the metric table structure from the Excel workbook

preview_df, preview_meta = load_metric_dataframe(
    route_id=1,
    metric_mode=METRIC_MODE,
    time_set=TIME_SET
)

print("Metric label:", preview_meta["metric_label"])
print("Higher is worse:", preview_meta["higher_is_worse"])
print()
display(
    preview_df[[
        "Segment ID",
        "New Segment ID",
        "Street Name",
        "Speed Limit(kph)",
        "harmonic_speed_kph",
        "sample_size",
        "metric_value"
    ]].head(10)
)

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\JupyterProjects\\tomtom_move\\data\\traffic_stats\\jobs_9031079_results_Worcester_test.xlsx'

In [ ]:
# Cell 5 - Inspect the first feature properties in each GeoJSON file
# Run this once before building the map.
# If match counts later are poor, use this output to adjust GEOJSON_NEW_SEGMENT_KEYS / GEOJSON_SEGMENT_KEYS.

for route_id, filename in ROUTE_FILE_MAP.items():
    geojson_path = ROUTE_DIR / filename

    with open(geojson_path, "r", encoding="utf-8") as f:
        gj = json.load(f)

    first_feature = gj["features"][0]
    props = first_feature.get("properties", {})

    print(f"\nRoute {route_id}: {filename}")
    print("Feature count:", len(gj["features"]))
    print("First feature property keys:")
    print(list(props.keys()))